# Support Triage Agent — Single Ticket Walkthrough

Traces four tickets through every node of the LangGraph pipeline
(`ingest -> sentiment_policy_check -> rag_retrieve -> draft_answer -> route ->
confidence_recheck -> hitl_gate`), one for each of the four possible routing
outcomes, by reading each ticket's persisted `GraphState`.


| Ticket | Scenario | Actual route this run |
|---|---|---|
| TCK-1004 | Clean, in-policy subscription-cancellation request | `AUTO_RESOLVE` |
| TCK-1002 | Repeat refund request within the abuse window | `ESCALATE` |
| TCK-1005 | Ticket missing a required field | `ASK_INFO` |
| TCK-1009 | Abusive/threatening message | `REFUSE` |

In [1]:
import sys, json, sqlite3
from pathlib import Path

# Works whether Jupyter's cwd is sample_run/ or the repo root.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config.settings import get_settings
from src.main import load_tickets

settings = get_settings()
tickets_by_id = {t.ticket_id: t for t in load_tickets(settings)}
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded {len(tickets_by_id)} tickets from {settings.tickets_path.relative_to(REPO_ROOT)}")

def load_result(ticket_id: str) -> dict:
    return json.loads((settings.results_dir / f"{ticket_id}.json").read_text(encoding="utf-8"))

def latest_reviewer_action(ticket_id: str) -> str | None:
    """The result JSON's reviewer_action is only a snapshot of hitl_gate's
    output the moment the graph run finished -- None whenever the ticket went
    into the async reviewer queue instead of being auto-approved. The reviews
    table in outputs/databases/triage.db gets updated in place once a human
    (or the CLI's --queue processor) actually reviews it, so pull the most
    recently updated row for this ticket from there instead."""
    conn = sqlite3.connect(settings.reviewer_db_path)
    try:
        row = conn.execute(
            "SELECT reviewer_action FROM reviews WHERE ticket_id = ? ORDER BY updated_at DESC LIMIT 1",
            (ticket_id,),
        ).fetchone()
    finally:
        conn.close()
    return row[0] if row else None

/Users/mahendarprakash/SupportTriageAgent/.supportvenv/lib/python3.14/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Repo root: /Users/mahendarprakash/SupportTriageAgent
Loaded 22 tickets from data/synthetic_tickets.json


## Node-by-node breakdown

One helper that prints a persisted `GraphState` grouped by the node that produced each field -- the same grouping `ARCHITECTURE.md` §4 documents the pipeline in.

In [2]:
def explain(ticket_id: str) -> dict:
    ticket = tickets_by_id[ticket_id]
    state = load_result(ticket_id)

    print(f"##### ticket (input) #####")
    print(json.dumps(ticket.model_dump(), indent=2))

    print(f"\n##### ingest #####")
    print(f"conversation_context: {state.get('conversation_context')!r}")

    print(f"\n##### sentiment_policy_check #####")
    for k in ["abuse_detected", "sentiment", "detected_category", "requires_more_info", "missing_fields"]:
        print(f"{k}: {state.get(k)}")

    print(f"\n##### rag_retrieve #####")
    chunks = state.get("retrieved_chunks") or []
    print(f"retrieval_attempts: {state.get('retrieval_attempts')}")
    for c in chunks:
        print(f"  [{c['score']:.3f}] {c['source']}: {c['text'][:90].strip()}...")

    print(f"\n##### draft_answer #####")
    print(f"groundedness_score (retrieval-similarity): {state.get('groundedness_score')}")
    print(f"fabricated_citations: {state.get('fabricated_citations')}")
    print(f"draft_reply:\n{state.get('draft_reply')}")

    print(f"\n##### route #####")
    print(f"route_decision: {state.get('route_decision')}")
    print(f"route_reason:   {state.get('route_reason')}")

    if state.get("llm_groundedness_score") is not None:
        print(f"\n##### confidence_recheck (only reached for AUTO_RESOLVE) #####")
        print(f"llm_groundedness_score:  {state.get('llm_groundedness_score')}")
        print(f"llm_groundedness_passed: {state.get('llm_groundedness_passed')}")
        print(f"unsupported_claims:      {state.get('unsupported_claims')}")

    print(f"\n##### hitl_gate #####")
    print(f"reviewer_action: {latest_reviewer_action(ticket_id)}  (review_id={state.get('review_id')})")

    return state

## TCK-1004 — clean, in-policy subscription-cancellation request

Actual route: **AUTO_RESOLVE**

In [3]:
_ = explain("TCK-1004")

##### ticket (input) #####
{
  "ticket_id": "TCK-1004",
  "customer_id": "CUST-004",
  "subject": "Cancel my subscription",
  "message": "Please cancel my monthly subscription (SUB-4410) before the next billing date.",
  "conversation_history": [],
  "priority": "low",
  "category": "subscription_cancellation",
  "order_id": null,
  "account_id": null,
  "subscription_id": "SUB-4410",
  "days_since_purchase": null,
  "previous_refund_request_count": 0,
  "days_since_last_refund_request": null
}

##### ingest #####
conversation_context: '(no conversation history)'

##### sentiment_policy_check #####
abuse_detected: False
sentiment: neutral
detected_category: subscription_cancellation
requires_more_info: False
missing_fields: []

##### rag_retrieve #####
retrieval_attempts: 1
  [0.607] subscription_policy.md: # Subscription Policy

## Billing and Renewal
- Monthly and annual subscriptions renew aut...
  [0.393] subscription_policy.md: ## Plan Changes
- Customers may switch plans at any t

## TCK-1002 — repeat refund request within the abuse window

Actual route: **ESCALATE**. The deterministic rule layer (`src/rules/refund_rules.py::check_repeat_request`) catches this before drafting even matters for the routing decision.

In [4]:
_ = explain("TCK-1002")

##### ticket (input) #####
{
  "ticket_id": "TCK-1002",
  "customer_id": "CUST-002",
  "subject": "Need refund again",
  "message": "I already requested a refund last month for order ORD-2091 and now I am asking again because I still have not received it.",
  "conversation_history": [
    {
      "role": "customer",
      "content": "I requested a refund last month."
    }
  ],
  "priority": "high",
  "category": "refund_request",
  "order_id": "ORD-2091",
  "account_id": null,
  "subscription_id": null,
  "days_since_purchase": 45,
  "previous_refund_request_count": 1,
  "days_since_last_refund_request": 30
}

##### ingest #####
conversation_context: 'CUSTOMER: I requested a refund last month.'

##### sentiment_policy_check #####
abuse_detected: False
sentiment: negative
detected_category: refund_request
requires_more_info: False
missing_fields: []

##### rag_retrieve #####
retrieval_attempts: 1
  [0.604] refund_policy.md: ## Repeated Refund Requests
- If a customer submits multiple r

## TCK-1005 — missing a required field

Actual route: **ASK_INFO**. `src/rules/required_fields.py` checks the category's required fields (e.g. `order_id` for a refund) against what's actually on the ticket.

In [5]:
_ = explain("TCK-1005")

##### ticket (input) #####
{
  "ticket_id": "TCK-1005",
  "customer_id": "CUST-005",
  "subject": "Cannot sign in",
  "message": "I cannot log in to my account and I need help accessing my profile.",
  "conversation_history": [],
  "priority": "high",
  "category": "login_access",
  "order_id": null,
  "account_id": null,
  "subscription_id": null,
  "days_since_purchase": null,
  "previous_refund_request_count": 0,
  "days_since_last_refund_request": null
}

##### ingest #####
conversation_context: '(no conversation history)'

##### sentiment_policy_check #####
abuse_detected: False
sentiment: neutral
detected_category: login_access
requires_more_info: True
missing_fields: ['account identifier', 'error description']

##### rag_retrieve #####
retrieval_attempts: 1
  [0.473] account_access_faq.md: # Account Access FAQ

## Common Issues
- Password reset: Customers can use the password re...
  [0.411] account_access_faq.md: ## Required Information
Before troubleshooting account access, re

## TCK-1009 — abusive/threatening message

Actual route: **REFUSE**. `src/rules/abuse_detection.py::detect_abuse` is a deterministic keyword scan that fires *before* any drafting LLM call -- the refusal text is a fixed template, not model output.

In [6]:
_ = explain("TCK-1009")

##### ticket (input) #####
{
  "ticket_id": "TCK-1009",
  "customer_id": "CUST-009",
  "subject": "You people are useless",
  "message": "This is absolutely infuriating. You people are useless and I'm sick of dealing with incompetent idiots. Fix this now or I'll make you regret ever taking my money.",
  "conversation_history": [],
  "priority": "high",
  "category": "abusive_content",
  "order_id": null,
  "account_id": null,
  "subscription_id": null,
  "days_since_purchase": null,
  "previous_refund_request_count": 0,
  "days_since_last_refund_request": null
}

##### ingest #####
conversation_context: '(no conversation history)'

##### sentiment_policy_check #####
abuse_detected: True
sentiment: abusive
detected_category: other
requires_more_info: True
missing_fields: ['issue description', 'account/order identifier']

##### rag_retrieve #####
retrieval_attempts: 1
  [0.216] troubleshooting_faq.md: # Troubleshooting FAQ

## Common Technical Issues
- If the app is not loading, ask the 

## Summary across all four

In [7]:
import pandas as pd

rows = []
for tid in ["TCK-1004", "TCK-1002", "TCK-1005", "TCK-1009"]:
    s = load_result(tid)
    rows.append({
        "ticket_id": tid,
        "category": tickets_by_id[tid].category,
        "route_decision": s["route_decision"],
        "route_reason": s["route_reason"],
        "groundedness_score": s.get("groundedness_score"),
        "llm_groundedness_score": s.get("llm_groundedness_score"),
        "reviewer_action": latest_reviewer_action(tid),
    })

pd.DataFrame(rows)

,ticket_id,category,route_decision,route_reason,groundedness_score,llm_groundedness_score,reviewer_action
0,TCK-1004,subscription_cancellation,AUTO_RESOLVE,policy_grounded_response,0.606715,1.0,APPROVED
1,TCK-1002,refund_request,ESCALATE,repeat_refund_request_within_window,0.604440,NaN,APPROVED
2,TCK-1005,login_access,ASK_INFO,missing_required_fields,0.473260,NaN,APPROVED
3,TCK-1009,abusive_content,REFUSE,abusive_content_detected,0.215538,NaN,APPROVED
